# S0 — Resolve data sources

Stage 0 of the Manhattan Sidewalk Shade Index pipeline.

Resolves every dataset in `CLAUDE.md` §3 against **live** catalogs — no hardcoded/guessed dataset IDs — verifies column names against the published data dictionary, and writes the resolved endpoints to `data/SOURCES.md`.

**Amendment to CLAUDE.md §3 (recorded in `docs/DECISIONS.md`):** the spec assumes sidewalks and LION are DCP-only, not on Socrata. Live investigation found both *are* registered on NYC Open Data (Socrata) — the DCP nyc.gov pages turned out to be JS-rendered shells a plain fetch can't read anyway, so this notebook resolves all five layers through the Socrata catalog API, handling three asset shapes it returns:
- **tabular** — a normal queryable dataset; verify columns directly.
- **map** — a visualization wrapper around a real dataset; follow its `modifyingViewUid` to the backing tabular dataset and verify columns there (this is what "Sidewalk" resolves to — the top catalog hit is the wrapper).
- **blobby** — a downloadable file (e.g. LION ships as a zipped shapefile, not a tabular dataset); no columns to validate, resolves to a direct blob download URL instead.

Does **not** download data — see `s1_ingest.ipynb`.

**Accept when:** every required layer (trees-primary, sidewalks, LION, borough boundary) resolves with status `success`. `street_trees_alt` is informational only (§3: "use only if the 2015 census proves unusable") and does not gate acceptance.

In [1]:
import sys
import json
import re
from datetime import datetime
from pathlib import Path
from typing import Any

import requests
import yaml

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config.yaml").exists() else Path.cwd().parent
config = yaml.safe_load(open(PROJECT_ROOT / "config.yaml"))
UA = {"User-Agent": "Mozilla/5.0"}
print(f"s0: Resolving data sources...\nTimestamp: {datetime.now().isoformat()}\n")

s0: Resolving data sources...
Timestamp: 2026-08-28T13:24:32.497415



## Socrata resolution

Search the live catalog by title, then branch on the returned asset shape (`tabular` / `map` / `blobby`) as described above.

In [2]:
def _view_metadata(dataset_id: str) -> dict:
    resp = requests.get(f"https://data.cityofnewyork.us/api/views/{dataset_id}", headers=UA, timeout=15)
    resp.raise_for_status()
    return resp.json()


def resolve_socrata_dataset(dataset_title: str, expected_columns: list[str]) -> dict[str, Any]:
    """Resolve a Socrata dataset from NYC Open Data by title via the live catalog API.

    Handles tabular datasets directly, follows 'map' visualization wrappers to their
    backing tabular dataset, and returns a blob-download URL for 'blobby' file assets
    (no column validation possible for those -- they are not tabular).
    """
    catalog_url = "https://api.us.socrata.com/api/catalog/v1"
    try:
        resp = requests.get(
            catalog_url,
            params={"search_context": "data.cityofnewyork.us", "q": dataset_title, "limit": 5},
            headers=UA,
            timeout=15,
        )
        resp.raise_for_status()
        results = resp.json().get("results", [])
        if not results:
            return {"status": "error", "message": f"No Socrata dataset found for title: {dataset_title}"}

        resource = results[0].get("resource", {})
        dataset_id = resource.get("id")
        name = resource.get("name")
        if not dataset_id:
            return {"status": "error", "message": f"Socrata result found but no dataset ID: {name}"}

        view = _view_metadata(dataset_id)

        # 'map' asset: a visualization wrapper -- follow to the real backing dataset
        if view.get("assetType") == "map" and view.get("modifyingViewUid"):
            dataset_id = view["modifyingViewUid"]
            view = _view_metadata(dataset_id)
            name = view.get("name", name)

        # 'blobby' asset: a downloadable file (e.g. a zipped shapefile), not tabular
        if view.get("viewType") == "blobby":
            blob_id = view.get("blobId")
            filename = view.get("blobFilename")
            if not blob_id:
                return {"status": "error", "message": f"Dataset {dataset_id} is a blob asset with no blobId"}
            return {
                "status": "success",
                "dataset_id": dataset_id,
                "name": name,
                "asset_kind": "blob",
                "url": f"https://data.cityofnewyork.us/api/views/{dataset_id}/files/{blob_id}?download=true&filename={filename}",
                "message": f"Blob file asset ({filename}, {view.get('blobFileSize', 0):,} bytes) -- not a queryable table, no columns to verify.",
            }

        columns = {c["name"]: c.get("dataTypeName", "unknown") for c in view.get("columns", []) if c.get("name")}
        missing = [c for c in expected_columns if c not in columns]
        status = "warning" if missing else "success"
        result = {
            "status": status,
            "dataset_id": dataset_id,
            "name": name,
            "asset_kind": "tabular",
            "url": f"https://data.cityofnewyork.us/resource/{dataset_id}.geojson",
            "columns": columns,
        }
        if missing:
            result["message"] = f"Dataset {dataset_id} missing expected columns: {missing}"
        return result

    except requests.RequestException as e:
        return {"status": "error", "message": f"Network error resolving {dataset_title}: {e}"}
    except (json.JSONDecodeError, KeyError) as e:
        return {"status": "error", "message": f"Parse error resolving {dataset_title}: {e}"}

## Run resolution for every source in CLAUDE.md §3

In [3]:
sources = {
    "street_trees_primary": {
        "title": "2015 Street Tree Census (Tree Data)",
        "expected_columns": ["tree_id", "status", "tree_dbh", "spc_latin", "latitude", "longitude"],
        "required": True,
    },
    "street_trees_alt": {
        "title": "Forestry Tree Points",
        "expected_columns": ["TPCondition", "DBH", "GenusSpecies"],
        "required": False,
    },
    "borough_boundary": {
        "title": "Borough Boundaries",
        "expected_columns": ["BoroCode", "BoroName", "the_geom"],
        "required": True,
    },
    "sidewalks": {
        "title": "NYC Planimetric Database Sidewalk",
        "expected_columns": ["the_geom", "FEAT_CODE", "STATUS"],
        "required": True,
    },
    "street_centerlines": {
        "title": "LION",
        "expected_columns": [],  # blob asset (zipped shapefile) -- no columns to check
        "required": True,
    },
}

resolutions: dict[str, dict] = {}
for key, info in sources.items():
    print(f"Resolving: {key}...")
    result = resolve_socrata_dataset(info["title"], info["expected_columns"])
    result["required"] = info["required"]
    resolutions[key] = result
    print(f"  status={result.get('status')} " + (result.get("message") or result.get("url", "")))

Resolving: street_trees_primary...


  status=success https://data.cityofnewyork.us/resource/uvpi-gqnh.geojson
Resolving: street_trees_alt...


  status=success https://data.cityofnewyork.us/resource/hn5i-inap.geojson
Resolving: borough_boundary...


  status=success https://data.cityofnewyork.us/resource/gthc-hcne.geojson
Resolving: sidewalks...


  status=success https://data.cityofnewyork.us/resource/52n9-sdep.geojson
Resolving: street_centerlines...


  status=success Blob file asset (nyclion.zip, 45,981,902 bytes) -- not a queryable table, no columns to verify.


## QA summary and write `data/SOURCES.md`

In [4]:
def print_qa_summary(resolutions: dict[str, dict]) -> tuple[int, int, int]:
    print("\n" + "=" * 70)
    print("S0 — Resolve sources: QA Summary")
    print("=" * 70)
    success = sum(1 for r in resolutions.values() if r["status"] == "success")
    warning = sum(1 for r in resolutions.values() if r["status"] == "warning")
    error = sum(1 for r in resolutions.values() if r["status"] == "error")
    print(f"Resolutions attempted: {len(resolutions)}")
    print(f"  success: {success}  warning: {warning}  error: {error}")
    for key, r in resolutions.items():
        if r["status"] != "success":
            flag = "REQUIRED" if r.get("required") else "optional"
            print(f"  [{r['status']}] {key} ({flag}): {r.get('message')}")
    print("=" * 70 + "\n")
    return success, warning, error


success, warning, error = print_qa_summary(resolutions)

sources_file = PROJECT_ROOT / "data" / "SOURCES.md"
resolution_summary = f"\n## Resolution Results\n\n**Resolved:** {datetime.now().isoformat()}\n"
for key, result in resolutions.items():
    resolution_summary += f"\n### {key}\n"
    resolution_summary += f"**Status:** {result.get('status')}\n"
    if result.get("dataset_id"):
        resolution_summary += f"**Dataset ID:** {result.get('dataset_id')}\n"
    if result.get("asset_kind"):
        resolution_summary += f"**Asset kind:** {result.get('asset_kind')}\n"
    if result.get("url"):
        resolution_summary += f"**URL:** {result.get('url')}\n"
    if result.get("message"):
        resolution_summary += f"**Message:** {result.get('message')}\n"
    if result.get("columns"):
        resolution_summary += f"**Columns:** {', '.join(result['columns'].keys())}\n"

existing = sources_file.read_text()
existing = re.split(r"\n## Resolution Results\n", existing)[0]
sources_file.write_text(existing + resolution_summary)
print(f"Updated {sources_file}")

required_errors = [k for k, r in resolutions.items() if r.get("required") and r["status"] == "error"]
if required_errors:
    print(f"\nFAILED — required sources unresolved: {required_errors}")
    print("Fix errors above before proceeding to S1. Do not guess IDs.")
else:
    print("\nS0 complete — all required sources resolved. Ready for S1 (ingest).")


S0 — Resolve sources: QA Summary
Resolutions attempted: 5
  success: 5  warning: 0  error: 0

Updated C:\Users\juanz\OneDrive\Desktop\Cursos\K3-MODERN-GIS\accelerator\accelerator\part4-cloud-capstone\4.4-coding-agents\data\SOURCES.md

S0 complete — all required sources resolved. Ready for S1 (ingest).
